# Walk through the hard case of the five-colour proof

*Adversarial Mathematics From First Principles*, [Part II, chapter 5](https://programmer.ie/books/math/14-chapter/) · [worked example](https://programmer.ie/examples/math/14-chapter/)

Run the cells in order. The program is a small, self-contained demonstration of the chapter's mathematics; the output saved with it is what it printed when this notebook was built.

Heawood's five-colour proof removes a vertex of degree at most five, colours
the rest, and puts the vertex back. The only hard case is five neighbours
with five different colours. The icosahedron is planar with every degree five,
so the case can be built on purpose. This program builds it and then follows
the proof: try one Kempe chain, and if it fails, use the one planarity
guarantees.

## Run it

Standard library only; nothing to install.

In [1]:
"""Maps, graphs and colourings: the hard case of Heawood's five-colour proof (standard library only).

The graph is the icosahedron: planar, and every vertex has degree 5. Remove the top
vertex v, colour the rest so that v's five neighbours get five different colours, and
then do what Heawood's proof does: swap one Kempe chain to free a colour for v.
"""
TOP, RING, LOW, BOTTOM = 0, [1, 2, 3, 4, 5], [6, 7, 8, 9, 10], 11
EDGES = ([(TOP, r) for r in RING] + [(RING[i], RING[(i + 1) % 5]) for i in range(5)]
         + [(RING[i], LOW[i]) for i in range(5)] + [(RING[i], LOW[(i + 1) % 5]) for i in range(5)]
         + [(LOW[i], LOW[(i + 1) % 5]) for i in range(5)] + [(BOTTOM, w) for w in LOW])
ADJ = {v: set() for v in range(12)}
for a, b in EDGES:
    ADJ[a].add(b)
    ADJ[b].add(a)

V, E, F = 12, len(EDGES), 20
print(f"icosahedron: V - E + F = {V} - {E} + {F} = {V - E + F}; E <= 3V - 6: {E} <= {3 * V - 6}")
print(f"every degree is {sorted({len(n) for n in ADJ.values()})}: the proof's vertex of degree <= 5 always exists")


def colour_rest(fixed):
    """Complete a proper 5-colouring of G - TOP that agrees with `fixed`, by search."""
    col, todo = dict(fixed), [u for u in sorted(ADJ) if u != TOP and u not in fixed]

    def go(i):
        if i == len(todo):
            return True
        for c in range(5):
            if all(col.get(w) != c for w in ADJ[todo[i]]):
                col[todo[i]] = c
                if go(i + 1):
                    return True
                del col[todo[i]]
        return False

    assert go(0)
    return col


col = colour_rest({r: i for i, r in enumerate(RING)})       # the hard case: five neighbours, five colours
print(f"\nG - v coloured; v's neighbours {RING} have colours {[col[r] for r in RING]}: no colour free for v")


def chain(start, c1, c2):
    seen, stack = {start}, [start]
    while stack:
        for w in ADJ[stack.pop()] - {TOP}:
            if col[w] in (c1, c2) and w not in seen:
                seen.add(w)
                stack.append(w)
    return seen


# Heawood: neighbours 1 and 3 are not adjacent. If they lie in one chain, that chain and v
# form a closed curve separating 2 from 4, so 2 and 4 cannot share a chain. Try 1-3, then 2-4.
for a, b in ((RING[0], RING[2]), (RING[1], RING[3])):
    ka = chain(a, col[a], col[b])
    print(f"neighbours {a} and {b} (colours {col[a]}, {col[b]}): {a}'s {col[a]}-{col[b]} chain is {sorted(ka)}; "
          f"contains {b}? {b in ka}")
    if b not in ka:
        ca, cb = col[a], col[b]
        for u in ka:
            col[u] = cb if col[u] == ca else ca
        print(f"  swap that chain: {a} becomes {col[a]}, so colour {ca} is free for v")
        col[TOP] = ca
        break
    print(f"  same chain: it closes a curve through v that separates {RING[1]} from {RING[3]}")

proper = all(col[x] != col[y] for x, y in EDGES)
print(f"\nfinal colouring {dict(sorted(col.items()))}")
print(f"proper: {proper}; colours used: {len(set(col.values()))}")

icosahedron: V - E + F = 12 - 30 + 20 = 2; E <= 3V - 6: 30 <= 30
every degree is [5]: the proof's vertex of degree <= 5 always exists

G - v coloured; v's neighbours [1, 2, 3, 4, 5] have colours [0, 1, 2, 3, 4]: no colour free for v
neighbours 1 and 3 (colours 0, 2): 1's 0-2 chain is [1, 3, 7, 8]; contains 3? True
  same chain: it closes a curve through v that separates 2 from 4
neighbours 2 and 4 (colours 1, 3): 2's 1-3 chain is [2]; contains 4? False
  swap that chain: 2 becomes 3, so colour 1 is free for v

final colouring {0: 1, 1: 0, 2: 3, 3: 2, 4: 3, 5: 4, 6: 1, 7: 2, 8: 0, 9: 1, 10: 0, 11: 3}
proper: True; colours used: 5


## What the output shows

Neighbours 1 and 3 lie in one chain of their two colours, so swapping it frees nothing.
That chain, with the vertex v, closes a curve separating neighbours 2 and 4,
so their chain cannot reach from one to the other: the swap on 2's chain is
safe and frees a colour for v. This is the step that works with five colours
and fails with four, where Kempe needed two swaps at once.

## Not shown

- One graph and one colouring: the program follows the proof's steps, it does not prove the theorem.
- The separating-curve argument (planarity, the Jordan curve theorem) is used, not checked: the program tries 2 and 4 because the proof says that pair must work.

## Challenge it

Examine the program's hard case on the icosahedron. Check Euler's formula,
the bound E ≤ 3V − 6 and the minimum degree. Explain why, if neighbours 1 and
3 share a red-green chain, neighbours 2 and 4 cannot share a chain in the other
two colours. Then say exactly where the same argument breaks with four
colours, as in Part I, chapter 6.